# Web Search Tool Test

Tests the `gateway__WebSearch` tool with 6 non-RSS URLs from the registry.
Simulates what the workflow agent does: sends a site-scoped query per URL per team.

**URLs tested (accessible, not outdated, no RSS):**
- Duke Blue Devils: 247Sports, On3
- Alabama Crimson Tide: On3 (2 forums)
- Arkansas Razorbacks: Hogville, Whole Hog Sports

In [1]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from dotenv import load_dotenv
load_dotenv()

from urllib.parse import urlparse
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp import MCPClient
from blog_search_agent import _get_m2m_token

# Setup MCP client for web search
gateway_url = os.environ.get('GATEWAY_URL')
gateway_token = os.environ.get('GATEWAY_TOKEN') or _get_m2m_token()

gateway_client = MCPClient(
    lambda: streamablehttp_client(
        url=gateway_url,
        headers={'Authorization': f'Bearer {gateway_token}'},
    ),
    prefix='gateway',
)

print(f'Gateway URL: {gateway_url}')
print('MCP client created')

Gateway URL: https://fast-stack-gateway-loi46lshmg.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp
MCP client created


In [2]:
# Define test URLs — accessible, non-outdated, no RSS feeds
test_urls = [
    {
        'team': 'Duke Blue Devils',
        'url': 'https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/',
        'platform': '247Sports',
    },
    {
        'team': 'Duke Blue Devils',
        'url': 'https://www.on3.com/boards/forums/duke-hoops-open-forum.322/',
        'platform': 'On3 (Rivals)',
    },
    {
        'team': 'Alabama Crimson Tide',
        'url': 'https://www.on3.com/boards/categories/alabama-crimson-tide.23/',
        'platform': 'On3 (XenForo-based)',
    },
    {
        'team': 'Alabama Crimson Tide',
        'url': 'https://www.on3.com/boards/forums/bol-round-table.24/',
        'platform': 'On3 / BOL Round Table'
    }
    # },
    # {
    #     'team': 'Arkansas Razorbacks',
    #     'url': 'https://hogville.net/',
    #     'platform': 'Hogville Fan Blog/News',
    # },
    # {
    #     'team': 'Arkansas Razorbacks',
    #     'url': 'https://forums.wholehogsports.com/',
    #     'platform': 'Whole Hog Sports (XenForo)',
    # },
]

print(f'{len(test_urls)} URLs to test:')
for t in test_urls:
    print(f"  {t['team']:25s} | {t['platform']:25s} | {t['url'][:60]}")

4 URLs to test:
  Duke Blue Devils          | 247Sports                 | https://247sports.com/college/duke/board/duke-blue-devils-ba
  Duke Blue Devils          | On3 (Rivals)              | https://www.on3.com/boards/forums/duke-hoops-open-forum.322/
  Alabama Crimson Tide      | On3 (XenForo-based)       | https://www.on3.com/boards/categories/alabama-crimson-tide.2
  Alabama Crimson Tide      | On3 / BOL Round Table     | https://www.on3.com/boards/forums/bol-round-table.24/


## Run web search for each URL

The workflow agent sends queries like: `site:{domain} {team} {event_keywords}`

We'll replicate that pattern here.

In [3]:
import asyncio

EVENT_KEYWORDS = "injury roster transfer schedule"

async def run_web_search(mcp_client, query):
    """Call the gateway WebSearch tool via MCP."""
    tools = mcp_client.list_tools_sync()
    # Find the web search tool
    ws_tool = None
    for t in tools:
        if 'WebSearch' in t.tool_name or 'web-search' in t.tool_name:
            ws_tool = t
            break
    if not ws_tool:
        # List available tools for debugging
        print(f'Available tools: {[t.tool_name for t in tools]}')
        return None
    
    result = mcp_client.call_tool_sync(
        tool_name=ws_tool.tool_name,
        tool_use_id='test',
        arguments={'query': query},
    )
    return result

print('Helper function defined. Running searches in next cell...')


Helper function defined. Running searches in next cell...


In [4]:
# Run web search for each URL using the MCP client context manager
results = []

with gateway_client:
    # List available tools first
    tools = gateway_client.list_tools_sync()
    tool_names = [t.tool_name for t in tools]
    print(f'Available gateway tools: {tool_names}\n')
    
    # Find the web search tool name
    ws_tool_name = None
    for name in tool_names:
        if 'WebSearch' in name or 'web-search' in name or 'websearch' in name.lower():
            ws_tool_name = name
            break
    
    if not ws_tool_name:
        print(f'ERROR: No web search tool found in: {tool_names}')
    else:
        # Strip the 'gateway_' prefix — MCP server knows the raw name
        raw_tool_name = ws_tool_name.removeprefix('gateway_')
        print(f'Using tool: {ws_tool_name} (raw: {raw_tool_name})\n')
        
        for entry in test_urls:
            team = entry['team']
            url = entry['url']
            domain = urlparse(url).netloc
            
            # Build query like the workflow agent does
            query = f'site:{domain} {team} {EVENT_KEYWORDS} last 30 days'
            # query = f'site:{url} {team} {EVENT_KEYWORDS} last 30 days'
            print(f'\n{"━"*80}')
            print(f'  Team:     {team}')
            print(f'  URL:      {url}')
            print(f'  Platform: {entry["platform"]}')
            print(f'  Query:    {query}')
            print(f'{"━"*80}')
            
            try:
                result = gateway_client.call_tool_sync(
                    name=raw_tool_name,
                    tool_use_id=f'test-{domain}',
                    arguments={'query': query},
                )
                
                # Pretty-print the full result
                if result and result.get('content'):
                    for content_block in result['content']:
                        if isinstance(content_block, dict) and 'text' in content_block:
                            text = content_block['text']
                            try:
                                parsed = json.loads(text)
                                # Pretty print the full JSON
                                print(f'\n  Status: {result.get("status", "unknown")}')
                                print(f'  Response type: {type(parsed).__name__}')
                                # Handle both: list [...] or dict {"id":..., "results":[...]}
                                items = parsed
                                if isinstance(parsed, dict) and 'results' in parsed:
                                    items = parsed['results']
                                elif not isinstance(parsed, list):
                                    print(json.dumps(parsed, indent=2)[:1000])
                                    items = []
                                
                                if isinstance(items, list) and items:
                                    print(f'  Items returned: {len(items)}')
                                    print()
                                    for i, item in enumerate(items, 1):
                                        print(f'  ┌─ Result {i} ─────────────────────────────────')
                                        print(f'  │ Title:   {item.get("title", "N/A")}')
                                        print(f'  │ URL:     {item.get("url") or "N/A"}')
                                        snippet = item.get("snippet") or item.get("text", "N/A")
                                        # Print full snippet
                                        for sline in str(snippet).split("\n")[:6]:
                                            print(f'  │ Snippet: {sline[:140]}')
                                        if item.get("publishedDate"):
                                            print(f'  │ Date:    {item["publishedDate"]}')
                                        print(f'  └────────────────────────────────────────────')
                                        print()
                                elif isinstance(parsed, dict):
                                    print(json.dumps(parsed, indent=2))
                                else:
                                    print(f'  {parsed}')
                            except json.JSONDecodeError:
                                print(f'\n  Raw text response:')
                                print(text)
                        else:
                            print(f'  Content block: {content_block}')
                else:
                    print(f'  No content in response')
                    print(f'  Full result: {result}')
                
                results.append({'entry': entry, 'query': query, 'result': result})
                
            except Exception as e:
                print(f'  ERROR: {e}')
                results.append({'entry': entry, 'query': query, 'error': str(e)})
        
        print(f'\n{"━"*80}')
        print(f'DONE: {len(results)} searches completed')



Available gateway tools: ['gateway_sample-tool-target___text_analysis_tool', 'gateway_web-search-tool___WebSearch']

Using tool: gateway_web-search-tool___WebSearch (raw: web-search-tool___WebSearch)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Team:     Duke Blue Devils
  URL:      https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/
  Platform: 247Sports
  Query:    site:247sports.com Duke Blue Devils injury roster transfer schedule last 30 days
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Status: success
  Response type: dict
  Items returned: 10

  ┌─ Result 1 ─────────────────────────────────
  │ Title:   Duke Blue Devils Basketball Message Board
  │ URL:     https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/
  │ Snippet: - 558 - 36,341 - VIP - 40 - Trevor Keels narrows list to three Sep 9th, 6:04 PM Topic Stats: 204 Posts, 16

---
## Direct web_fetch on the same URLs

Call `web_fetch` directly on each URL (no web search first).
This fetches the page HTML, strips it to text, and sends it to Haiku 4.5 for event extraction.

**Comparison goal:** Does web_fetch on the base URL find events that web search missed?

In [5]:
from datetime import datetime, timedelta, timezone
from tools.web_fetch import web_fetch

# Use a 7-day window so we have a reasonable chance of finding events
now = datetime.now(timezone.utc)
dt_start = (now - timedelta(days=7)).isoformat(timespec='seconds')
dt_end = now.isoformat(timespec='seconds')

print(f'Datetime window: {dt_start} -> {dt_end}')
print(f'Event types: INJURY, ROSTER, SCHEDULE_CHANGE')
print()

web_fetch_results = []

for entry in test_urls:
    team = entry['team']
    url = entry['url']
    
    print(f'\n{"━"*80}')
    print(f'  Team:     {team}')
    print(f'  URL:      {url}')
    print(f'  Platform: {entry["platform"]}')
    print(f'{"━"*80}')
    
    result = await web_fetch(
        url=url,
        team=team,
        datetime_start=dt_start,
        datetime_end=dt_end,
        event_types='INJURY,ROSTER,SCHEDULE_CHANGE',
        sport='NCAA Men\'s Basketball',
    )
    
    print(f'  error:            {result.get("error")}')
    print(f'  extraction_error: {result.get("extraction_error")}')
    print(f'  events found:     {len(result.get("events", []))}')
    
    if result.get('events'):
        print()
        for i, ev in enumerate(result['events'], 1):
            print(f'  ┌─ Event {i} ─────────────────────────────────')
            print(f'  │ type:       {ev.get("event_type")}')
            print(f'  │ player:     {ev.get("player_name", "N/A")}')
            print(f'  │ summary:    {ev.get("summary")}')
            print(f'  │ post_date:  {ev.get("blog_post_date")}')
            print(f'  │ source_url: {ev.get("source_url")}')
            print(f'  │ excerpt:    {ev.get("excerpt")}')
            print(f'  └────────────────────────────────────────────')
            print()
    
    web_fetch_results.append({'entry': entry, 'result': result})

print(f'\n{"━"*80}')
total_events = sum(len(r['result'].get('events', [])) for r in web_fetch_results)
print(f'DONE: {len(web_fetch_results)} URLs fetched, {total_events} total events extracted')

[WEB_FETCH] url=https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/ error=403 Client Error: Forbidden for url: https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/


Datetime window: 2026-07-23T11:41:40+00:00 -> 2026-07-30T11:41:40+00:00
Event types: INJURY, ROSTER, SCHEDULE_CHANGE


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Team:     Duke Blue Devils
  URL:      https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/
  Platform: 247Sports
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  error:            403 Client Error: Forbidden for url: https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/
  extraction_error: None
  events found:     0

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Team:     Duke Blue Devils
  URL:      https://www.on3.com/boards/forums/duke-hoops-open-forum.322/
  Platform: On3 (Rivals)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  error:            None
  extraction_error: None
  events found:     1

  ┌─ Ev

---
## Workflow agent with web search + web_fetch (agent decides)

Now let the workflow Sonnet agent handle the same URLs — it gets both `gateway__WebSearch`
and `web_fetch` tools and decides:
1. What search queries to run
2. Which search results are worth fetching
3. What to extract from fetched pages

**Comparison goal:** Does the agent (web search → pick URLs → web_fetch) find more/better
events than direct web_fetch alone?

In [6]:
from blog_search_agent import _create_workflow_agent, _build_non_rss_workflow_message
from blog_search_agent import extract_json_from_response, _parse_and_validate_with_retry
from models import EVENT_TYPES, BlogSearchResponse
from urllib.parse import urlparse

# Group test_urls by team for the agent
teams = {}
for entry in test_urls:
    teams.setdefault(entry['team'], []).append(entry)

print(f'Testing {len(teams)} teams with workflow agent:')
for team, entries in teams.items():
    print(f'  {team}: {len(entries)} URLs')
print()

Testing 2 teams with workflow agent:
  Duke Blue Devils: 2 URLs
  Alabama Crimson Tide: 2 URLs



In [7]:
# Run the workflow agent for each team
agent_results = {}

for team, entries in teams.items():
    print(f'\n{"━"*80}')
    print(f'  WORKFLOW AGENT: {team}')
    print(f'  URLs: {len(entries)}')
    print(f'{"━"*80}')
    
    # Build the blog list in the format the workflow expects
    non_rss_blogs = [{'url': e['url']} for e in entries]
    
    # Build the message the agent will see
    message = _build_non_rss_workflow_message(
        non_rss_blogs=non_rss_blogs,
        team=team,
        sport='NCAA Men\'s Basketball',
        events=['INJURY', 'ROSTER', 'SCHEDULE_CHANGE'],
        datetime_start=dt_start,
        datetime_end=dt_end,
    )
    
    print(f'\n  Agent message:')
    print(f'  {"─"*70}')
    for line in message.split('\n'):
        print(f'  │ {line}')
    print(f'  {"─"*70}')
    
    # Create and run the workflow agent
    agent = _create_workflow_agent(debug=True)
    result = agent(message)
    response_text = str(result)
    
    # Parse and validate
    validated = _parse_and_validate_with_retry(agent, response_text, debug=False)
    
    if isinstance(validated, BlogSearchResponse):
        events = [r.model_dump() for r in validated.results]
        print(f'\n  ✅ Agent found {len(events)} events:')
        for i, ev in enumerate(events, 1):
            print(f'    {i}. [{ev["event_type"]}] {ev.get("player_name", "")} — {ev["summary"][:80]}')
            print(f'       source: {ev["source_url"][:70]}')
            print(f'       date: {ev.get("blog_post_date")}')
            print()
        agent_results[team] = events
    else:
        print(f'\n  ❌ Agent failed: {validated}')
        agent_results[team] = []


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WORKFLOW AGENT: Duke Blue Devils
  URLs: 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Agent message:
  ──────────────────────────────────────────────────────────────────────
  │ Team: Duke Blue Devils
  │ Sport: NCAA Men's Basketball
  │ Event types to detect: INJURY, ROSTER, SCHEDULE_CHANGE
  │ Date range: 2026-07-23T11:41:40+00:00 to 2026-07-30T11:41:40+00:00
  │ Parameters to pass to web_fetch: team=Duke Blue Devils, datetime_start=2026-07-23T11:41:40+00:00, datetime_end=2026-07-30T11:41:40+00:00, event_types=INJURY,ROSTER,SCHEDULE_CHANGE, sport=NCAA Men's Basketball
  │ 
  │ URLs without RSS (2):
  │   - https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/ (domain: 247sports.com)
  │   - https://www.on3.com/boards/forums/duke-hoops-open-forum.322/ (domain: www.on3.com)
  │ 
  │ NOTE: RSS feeds have already been processed 

[WEB_FETCH] url=https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/ error=403 Client Error: Forbidden for url: https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/



  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch


  🔧 TOOL CALL: web_fetch

The on3 board returned a ROSTER event but with a date of 2026-07-20, which is outside the requested date range (2026-07-23 to 2026-07-30). All confirmed in-range events have been collected. Here is the final aggregated JSON output:

{
  "results": [
    {
      "sport": "NCAA Men's Basketball",
      "team": "Duke Blue Devils",
      "event_type": "SCHEDULE_CHANGE",
      "player_name": null,
      "excerpt": "The Red Raiders are replacing Michigan, who was originally scheduled to face the Blue Devils. A partnership dispute between Fox and Amazon Prime led to the schedule change.",
      "summary": "Duke's scheduled game against Michigan has been replaced with Texas T

---
## Comparison: Original URLs vs Discovered URLs & Events

For each method, compare:
- **Original URLs**: the 6 URLs from the registry (the ones we started with)
- **New URLs**: URLs discovered by web search that are NOT in the original set
- **Events**: how many events were extracted from each group

In [8]:
# Build the set of original URLs for comparison
original_urls = set(entry['url'] for entry in test_urls)
original_domains = {urlparse(u).netloc for u in original_urls}

print(f'Original URLs ({len(original_urls)}):')
for u in sorted(original_urls):
    print(f'  {u}')
print(f'\nOriginal domains: {sorted(original_domains)}')

Original URLs (4):
  https://247sports.com/college/duke/board/duke-blue-devils-basketball-message-board-101614/
  https://www.on3.com/boards/categories/alabama-crimson-tide.23/
  https://www.on3.com/boards/forums/bol-round-table.24/
  https://www.on3.com/boards/forums/duke-hoops-open-forum.322/

Original domains: ['247sports.com', 'www.on3.com']


In [9]:
# ─── Analyze Web Search results ───
# From the web search results, extract all URLs returned and classify as original vs new

ws_original_urls = set()
ws_new_urls = set()
ws_all_result_urls = []

for r in results:
    if r.get('result') and r['result'].get('content'):
        for content_block in r['result']['content']:
            if isinstance(content_block, dict) and 'text' in content_block:
                try:
                    parsed = json.loads(content_block['text'])
                    
                    # Handle both formats: list [...] or dict {"id":..., "results":[...]}
                    items = parsed
                    if isinstance(parsed, dict) and 'results' in parsed:
                        items = parsed['results']
                    elif not isinstance(parsed, list):
                        items = []
                    
                    for item in items:
                        result_url = item.get('url', '') or ''
                        if not result_url:
                            continue  # skip items with null/empty URL
                        ws_all_result_urls.append({
                            'url': result_url,
                            'title': item.get('title', ''),
                            'snippet': item.get('snippet') or item.get('text', ''),
                            'published_date': item.get('publishedDate', ''),
                            'team': r['entry']['team'],
                            'query': r['query'],
                        })
                        if result_url in original_urls:
                            ws_original_urls.add(result_url)
                        else:
                            ws_new_urls.add(result_url)
                except json.JSONDecodeError:
                    pass

print('═' * 80)
print('  WEB SEARCH RESULTS — URL Analysis')
print('═' * 80)
print(f'  Total result URLs returned:  {len(ws_all_result_urls)}')
print(f'  Unique URLs matching originals: {len(ws_original_urls)}')
print(f'  New URLs discovered:            {len(ws_new_urls)}')
print(f'\n  New URLs discovered:')
for u in sorted(ws_new_urls):
    print(f'    {u[:120]}')


════════════════════════════════════════════════════════════════════════════════
  WEB SEARCH RESULTS — URL Analysis
════════════════════════════════════════════════════════════════════════════════
  Total result URLs returned:  35
  Unique URLs matching originals: 1
  New URLs discovered:            25

  New URLs discovered:
    https://247sports.com/college/duke/board/101618/Contents/dwb-podcast-team-news-149443776/?page=2
    https://247sports.com/college/duke/board/101618/Contents/the-boykin-factor-144380941/?page=1
    https://247sports.com/college/duke/board/102415/Contents/duke-cornerback-josh-blackwell-out-indefinitely-after-surgery-1
    https://247sports.com/college/duke/board/duke-blue-devils-football-message-board-101615/
    https://247sports.com/college/duke/board/duke-blue-devils-message-board-forum-59461/
    https://247sports.com/college/duke/board/duke-blue-devils-womens-basketball-message-board-101618/
    https://247sports.com/college/duke/board/pascal-field-house-

In [10]:
# ─── Analyze Direct web_fetch results ───
# web_fetch only uses original URLs (no discovery), so all events come from originals

wf_events_by_team = {}
wf_total_events = 0

for r in web_fetch_results:
    team = r['entry']['team']
    events = r['result'].get('events', [])
    wf_events_by_team.setdefault(team, {'original_events': 0, 'original_urls': [], 'new_events': 0, 'new_urls': []})
    wf_events_by_team[team]['original_events'] += len(events)
    wf_events_by_team[team]['original_urls'].append(r['entry']['url'])
    wf_total_events += len(events)

print('═' * 80)
print('  DIRECT WEB_FETCH — Events from Original URLs only')
print('═' * 80)
print(f'  Total events: {wf_total_events} (all from original URLs, no new URLs discovered)')
for team, data in wf_events_by_team.items():
    print(f'    {team}: {data["original_events"]} events from {len(data["original_urls"])} URLs')

════════════════════════════════════════════════════════════════════════════════
  DIRECT WEB_FETCH — Events from Original URLs only
════════════════════════════════════════════════════════════════════════════════
  Total events: 1 (all from original URLs, no new URLs discovered)
    Duke Blue Devils: 1 events from 2 URLs
    Alabama Crimson Tide: 0 events from 2 URLs


In [11]:
# ─── Analyze Agent results ───
# The agent uses web search to find URLs then web_fetch on promising ones.
# Events have source_url — classify as original vs discovered

agent_events_original = {}  # team -> list of events from original URLs
agent_events_new = {}       # team -> list of events from discovered URLs
agent_new_urls_used = {}    # team -> set of new URLs the agent fetched

for team, events in agent_results.items():
    agent_events_original[team] = []
    agent_events_new[team] = []
    agent_new_urls_used[team] = set()
    
    for ev in events:
        source_url = ev.get('source_url', '')
        if source_url in original_urls:
            agent_events_original[team].append(ev)
        else:
            agent_events_new[team].append(ev)
            agent_new_urls_used[team].add(source_url)

print('═' * 80)
print('  AGENT MODE — Events from Original vs Discovered URLs')
print('═' * 80)
for team in teams:
    orig = agent_events_original.get(team, [])
    new = agent_events_new.get(team, [])
    new_urls = agent_new_urls_used.get(team, set())
    print(f'\n  {team}:')
    print(f'    From original URLs: {len(orig)} events')
    print(f'    From new URLs:      {len(new)} events ({len(new_urls)} new URLs)')
    if new_urls:
        for u in sorted(new_urls):
            print(f'      → {u[:90]}')

════════════════════════════════════════════════════════════════════════════════
  AGENT MODE — Events from Original vs Discovered URLs
════════════════════════════════════════════════════════════════════════════════

  Duke Blue Devils:
    From original URLs: 0 events
    From new URLs:      4 events (4 new URLs)
      → https://dukewire.usatoday.com/story/sports/college/duke/mens-basketball/2026/07/24/duke-an
      → https://sports.yahoo.com/articles/duke-michigan-rematch-basketball-turns-135808827.html
      → https://www.si.com/college/duke/blue-devils-basketball-replaces-michigan-familiar-non-con-
      → https://www.si.com/college/duke/blue-devils-how-michigan-cancellation-affects-basketball-s

  Alabama Crimson Tide:
    From original URLs: 0 events
    From new URLs:      10 events (5 new URLs)
      → https://rolltide.com/archives
      → https://rolltidewire.usatoday.com/story/sports/college/crimson-tide/mens-basketball/2026/0
      → https://www.on3.com/teams/alabama-crimso

In [15]:
# ─── Final Comparison Table with URLs and Snippets ───

print('\n' + '━' * 100)
print('  FINAL COMPARISON TABLE')
print('━' * 100)
print(f'  Time window: {dt_start} → {dt_end}')
print(f'  Original URLs: {len(original_urls)}')
print()

# Header
print(f'  {"Method":<28s} │ {"Orig URLs":<10s} │ {"Orig Evts":<10s} │ {"New URLs":<10s} │ {"New Evts":<10s} │ {"Total Evts"}')
print(f'  {"─"*28}┼{"─"*12}┼{"─"*12}┼{"─"*12}┼{"─"*12}┼{"─"*11}')

# Row 1: Web Search (snippets only — no event extraction)
print(f'  {"Web Search (snippets)":<28s} │ {len(ws_original_urls):<10d} │ {"N/A":<10s} │ {len(ws_new_urls):<10d} │ {"N/A":<10s} │ {"N/A"}')

# Row 2: Direct web_fetch
wf_orig_urls_count = len(original_urls)
print(f'  {"Direct web_fetch":<28s} │ {wf_orig_urls_count:<10d} │ {wf_total_events:<10d} │ {0:<10d} │ {0:<10d} │ {wf_total_events}')

# Row 3: Agent (web search + web_fetch)
agent_orig_total = sum(len(v) for v in agent_events_original.values())
agent_new_total = sum(len(v) for v in agent_events_new.values())
agent_new_urls_total = sum(len(v) for v in agent_new_urls_used.values())
agent_total = agent_orig_total + agent_new_total
print(f'  {"Agent (search + fetch)":<28s} │ {wf_orig_urls_count:<10d} │ {agent_orig_total:<10d} │ {agent_new_urls_total:<10d} │ {agent_new_total:<10d} │ {agent_total}')

print(f'  {"─"*28}┴{"─"*12}┴{"─"*12}┴{"─"*12}┴{"─"*12}┴{"─"*11}')

# ─── Web Search: New URLs discovered with snippets ───
print(f'\n\n{"━"*100}')
print(f'  WEB SEARCH — New URLs Discovered (with snippets)')
print(f'{"━"*100}')

# Group by team
ws_new_by_team = {}
for item in ws_all_result_urls:
    if item['url'] not in original_urls:
        ws_new_by_team.setdefault(item['team'], []).append(item)

for team, items in ws_new_by_team.items():
    print(f'\n  ┌─ {team} ({len(items)} new URLs from web search)')
    print(f'  │')
    seen = set()
    for item in items:
        if item['url'] in seen:
            continue
        seen.add(item['url'])
        print(f'  │  URL:     {item["url"]}')
        print(f'  │  Title:   {item["title"]}')
        print(f'  │  Snippet: {item["snippet"]}')
        print(f'  │')
    print(f'  └{"─"*70}')

# ─── Direct web_fetch: Events from original URLs ───
print(f'\n\n{"━"*100}')
print(f'  DIRECT WEB_FETCH — Events from Original URLs')
print(f'{"━"*100}')

for r in web_fetch_results:
    team = r['entry']['team']
    url = r['entry']['url']
    events = r['result'].get('events', [])
    print(f'\n  ┌─ {team}')
    print(f'  │  URL: {url}')
    print(f'  │  Events: {len(events)}')
    if events:
        for ev in events:
            print(f'  │    • [{ev.get("event_type")}] {ev.get("player_name", "")} — {ev.get("summary", "")[:90]}')
    else:
        print(f'  │    (no events extracted)')
    print(f'  └{"─"*70}')

# ─── Agent: Events from original vs new URLs ───
print(f'\n\n{"━"*100}')
print(f'  AGENT MODE — Events by Source URL')
print(f'{"━"*100}')

for team in teams:
    orig_events = agent_events_original.get(team, [])
    new_events = agent_events_new.get(team, [])
    new_urls = agent_new_urls_used.get(team, set())
    
    print(f'\n  ┌─ {team}')
    print(f'  │')
    print(f'  │  FROM ORIGINAL URLs ({len(orig_events)} events):')
    if orig_events:
        for ev in orig_events:
            print(f'  │    • [{ev.get("event_type")}] {ev.get("player_name", "")} — {ev.get("summary", "")[:80]}')
            print(f'  │      URL: {ev.get("source_url", "")}')
    else:
        print(f'  │    (none)')
    print(f'  │')
    print(f'  │  FROM NEW URLs ({len(new_events)} events, {len(new_urls)} new URLs):')
    if new_events:
        for ev in new_events:
            print(f'  │    • [{ev.get("event_type")}] {ev.get("player_name", "")} — {ev.get("summary", "")[:80]}')
            print(f'  │      URL: {ev.get("source_url", "")}')
    else:
        print(f'  │    (none)')
    print(f'  └{"─"*70}')

# ─── Conclusions ───
print(f'\n\n{"━"*100}')
print('  CONCLUSIONS')
print(f'{"━"*100}')
print(f'')
print(f'  1. WEB SEARCH DISCOVERY:')
print(f'     • Web search discovered {len(ws_new_urls)} new URLs beyond the original {len(original_urls)} registry URLs.')
print(f'     • These are raw, unfiltered search results — not all will contain relevant events.')
print(f'')
print(f'  2. AGENT APPROACH (search → select → fetch):')
print(f'     • The agent performed web searches that surfaced many candidate URLs (same pool as above).')
print(f'     • It then SELECTIVELY fetched only {agent_new_urls_total} URLs that appeared promising')
print(f'       based on titles/snippets — narrowing {len(ws_new_urls)} candidates down to {agent_new_urls_total}.')
print(f'     • From those {agent_new_urls_total} fetched URLs, it extracted {agent_new_total} events.')
print(f'     • This is the key value: the agent acts as a filter, saving LLM extraction cost')
print(f'       by only fetching pages likely to contain relevant events.')
print(f'')
print(f'  3. DIRECT WEB_FETCH (no search, just fetch original URLs):')
print(f'     • Fetching the {len(original_urls)} original registry URLs directly yielded {wf_total_events} events.')
if wf_total_events < agent_total:
    print(f'     • The agent found MORE events ({agent_total} vs {wf_total_events}) because web search')
    print(f'       discovers article-level URLs not reachable from base forum/homepage URLs.')
elif wf_total_events == agent_total:
    print(f'     • Same event count as agent — but agent may find different/better sourced events.')
else:
    print(f'     • More events than agent ({wf_total_events} vs {agent_total}) — base URLs already had content.')
print(f'')
print(f'  SUMMARY:')
print(f'     Web search (raw)    → {len(ws_new_urls)} new URLs discovered (broad, unfiltered)')
print(f'     Agent (selective)   → {agent_new_urls_total} URLs fetched → {agent_new_total} new events (targeted, cost-efficient)')
print(f'     Direct web_fetch    → {wf_total_events} events from {len(original_urls)} original URLs (no discovery)')
print(f'{"━"*100}')




━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FINAL COMPARISON TABLE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Time window: 2026-07-23T11:41:40+00:00 → 2026-07-30T11:41:40+00:00
  Original URLs: 4

  Method                       │ Orig URLs  │ Orig Evts  │ New URLs   │ New Evts   │ Total Evts
  ────────────────────────────┼────────────┼────────────┼────────────┼────────────┼───────────
  Web Search (snippets)        │ 1          │ N/A        │ 25         │ N/A        │ N/A
  Direct web_fetch             │ 4          │ 1          │ 0          │ 0          │ 1
  Agent (search + fetch)       │ 4          │ 0          │ 9          │ 14         │ 14
  ────────────────────────────┴────────────┴────────────┴────────────┴────────────┴───────────


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WEB SEARCH — New URLs Discov